# 可视化训练日志 - 训练过程各类别评估指标

本模块用于深入解析模型训练期间生成的 `.log` 纯文本日志文件，利用字符串动态匹配技术提取各类别（如红肉、绿皮等）在验证集上的单科成绩，并独立绘制高分辨率变化曲线。

## 1. 环境准备与全局配置

本节负责锁定绝对工作路径，并挂载中文字体，确保后续在各类别评估指标的可视化图表中，中文字符（如类名）能够正常渲染。

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 1. 锁定绝对工作目录
WORK_DIR = '/root/LearningMMSegmentation/mmsegmentation'
os.chdir(WORK_DIR)

# 2. 配置 Matplotlib 全局中文字体
FONT_PATH = '/root/LearningMMSegmentation/SimHei.ttf'
my_font = fm.FontProperties(fname=FONT_PATH)
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False

print(f"✅ 环境初始化完成。当前工作目录: {os.getcwd()}")

✅ 环境初始化完成。当前工作目录: /root/LearningMMSegmentation/mmsegmentation


## 2. 定位原始日志文件 (.log)

通过路径拼接，定位包含验证集详细评估表格的 `.log` 纯文本文件。必须确保任务名称与时间戳与最新一次训练保持一致。

In [2]:
# 1. 指定训练任务名称与时间戳
# 【重点关注】此处已更新为 F1 导出的自定义配置文件名
LOG_TASK_NAME = 'CustomDataset_UNet_pipeline' 
# 【需手动修改】请前往 work_dirs/CustomDataset_UNet_pipeline 目录下查看最新生成的文件夹名称，并替换下方的时间戳
LOG_TIMESTAMP = '20260403_170557' 

# 2. 拼接 .log 文件的完整路径
log_file_path = os.path.join('work_dirs', LOG_TASK_NAME, LOG_TIMESTAMP, f'{LOG_TIMESTAMP}.log')

if not os.path.exists(log_file_path):
    raise FileNotFoundError(f"❌ 找不到日志文件，请核对时间戳: {log_file_path}")
else:
    print(f"✅ 成功定位日志文件: {log_file_path}")

✅ 成功定位日志文件: work_dirs/CustomDataset_UNet_pipeline/20260403_170557/20260403_170557.log


## 3. 日志解析与类别数据提取

采用基于分隔符 `|` 的动态切割法逐行扫描日志文本。通过捕获 `Iter(train)` 锚定当前训练步数，随后安全提取紧跟其后的评估表格数据。此方法内置了安全浮点转换机制，能够有效规避训练初期因模型未预测出某类别而输出 `nan` 导致的解析崩溃。

In [3]:
# 1. 严格对齐日志中实际打印的类别英文名称
classes = ['background', 'red', 'green', 'white', 'seed-black', 'seed-white']
metrics_keys = ['IoU', 'Acc', 'Dice', 'Fscore', 'Precision', 'Recall']
metrics_data = {cls: {key: [] for key in ['step'] + metrics_keys} for cls in classes}

# 2. 执行强健的日志解析逻辑
current_step = None

with open(log_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        # 抓取最近一次打印的训练步数作为评估锚点
        train_step_match = re.search(r'Iter\(train\)\s+\[\s*(\d+)/\d+\]', line)
        if train_step_match:
            current_step = int(train_step_match.group(1))
        
        # 利用 '|' 分割表格行，无视空格排版差异与前缀字符
        if '|' in line and current_step is not None:
            parts = [p.strip() for p in line.split('|')]
            
            # 确认该行为包含数据的有效表格行
            if len(parts) > 2:
                class_name = parts[1]
                if class_name in classes:
                    
                    def safe_float(val_str):
                        """安全浮点数转换，遇到 nan 时返回 np.nan 以兼容 Matplotlib 绘图"""
                        try:
                            return float(val_str)
                        except ValueError:
                            return np.nan
                            
                    metrics_data[class_name]['step'].append(current_step)
                    metrics_data[class_name]['IoU'].append(safe_float(parts[2]))
                    metrics_data[class_name]['Acc'].append(safe_float(parts[3]))
                    
                    # 长度保护机制，兼容不同评估配置下的隐式列
                    metrics_data[class_name]['Dice'].append(safe_float(parts[4]) if len(parts) > 4 else np.nan)
                    metrics_data[class_name]['Fscore'].append(safe_float(parts[5]) if len(parts) > 5 else np.nan)
                    metrics_data[class_name]['Precision'].append(safe_float(parts[6]) if len(parts) > 6 else np.nan)
                    metrics_data[class_name]['Recall'].append(safe_float(parts[7]) if len(parts) > 7 else np.nan)

print("✅ 日志全量提取完成。")
print(f"📊 数据质检：成功提取 红肉 (red) 验证节点数 -> {len(metrics_data['red']['step'])}")

✅ 日志全量提取完成。
📊 数据质检：成功提取 红肉 (red) 验证节点数 -> 10


## 4. 目录准备与绘图配置

创建统一的图表保存路径，并重新定义实际类别名称的中英文映射关系，确保最终生成的图例清晰准确。

In [4]:
# 1. 建立保存目录
SAVE_DIR = '/root/LearningMMSegmentation/Learning/Evaluation Criteria by Category Pictures'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"📁 图表保存路径已确认: {SAVE_DIR}\n")

# 2. 终极修正：重新映射日志类名为中文显示名称
name_map = {
    'background': '背景 (background)',
    'red': '红肉 (red)',
    'green': '绿皮 (green)',
    'white': '白皮 (white)',
    'seed-black': '黑籽 (seed-black)',
    'seed-white': '白籽 (seed-white)'
}

# 3. 统一样式函数
def get_line_arg():
    """返回统一的绘图样式参数"""
    return {'linewidth': 2, 'marker': 'o', 'markersize': 5}

📁 图表保存路径已确认: /root/LearningMMSegmentation/Learning/Evaluation Criteria by Category Pictures



## 5. 各类别评估指标自动可视化

遍历预设的所有类别，为每个类别独立绘制其 `IoU`, `Acc`, `Dice` 等指标随训练步数变化的曲线，并自动保存至指定目录。

In [5]:
# 遍历所有类别进行绘图
for target_class in classes:
    data = metrics_data[target_class]
    
    # 仅当该类别提取到有效步数时进行绘图
    if data['step']:
        plt.figure(figsize=(12, 6))
        
        # 绘制该类别下的所有指标曲线
        for key in metrics_keys:
            # Matplotlib 会自动忽略列表中的 np.nan，保证曲线连续性
            plt.plot(data['step'], data[key], label=key, **get_line_arg())

        plt.xlabel('Step', fontsize=12)
        plt.ylabel('Metrics Score (%)', fontsize=12)
        plt.title(f'类别 [{name_map[target_class]}] 测试集评估指标演变', fontproperties=my_font, fontsize=16)
        
        # 设置 Y 轴范围为 0-105，留出图例空间
        plt.ylim([0, 105])
        plt.grid(True, linestyle='--', alpha=0.6)
        
        # 将图例放置在外部右下角避免遮挡曲线
        plt.legend(bbox_to_anchor=(1.12, 0), loc='lower right', borderaxespad=0.)

        # 导出高清图片
        save_path = os.path.join(SAVE_DIR, f'Evaluation_Metrics_{target_class}.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close() # 关闭画布，避免在 Notebook 中堆叠过长
        
        print(f"🎉 已成功生成并保存图表: Evaluation_Metrics_{target_class}.png")
    else:
        print(f"⚠️ 类别 {target_class} 未提取到有效数据，已跳过绘图。")

print(f"\n✅ 全部类别独立可视化执行完毕！请前往 {SAVE_DIR} 查看结果。")

findfont: Font family ['sans-serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Font family ['sans-serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei


🎉 已成功生成并保存图表: Evaluation_Metrics_background.png
🎉 已成功生成并保存图表: Evaluation_Metrics_red.png
🎉 已成功生成并保存图表: Evaluation_Metrics_green.png
🎉 已成功生成并保存图表: Evaluation_Metrics_white.png
🎉 已成功生成并保存图表: Evaluation_Metrics_seed-black.png
🎉 已成功生成并保存图表: Evaluation_Metrics_seed-white.png

✅ 全部类别独立可视化执行完毕！请前往 /root/LearningMMSegmentation/Learning/Evaluation Criteria by Category Pictures 查看结果。
